# 06-8. 로컬 Echo 프로젝트 검증 예제

## Goal

- 교안의 길이 접두사 프로토콜 모듈을 재사용합니다.
- 포트를 열지 않고 왕복 메시지를 확인합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

저장소 루트에서 실행합니다. `socketpair()`만 사용하며 외부 연결은 없습니다.


## Steps

### 프로토콜 모듈로 왕복 메시지 확인

실제 서버와 같은 `send_message()`·`receive_message()`를 로컬 소켓 쌍에 적용합니다.


In [1]:
from pathlib import Path
import socket
import sys
import threading


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "examples/06-network-echo/protocol.py").is_file():
            return candidate
    raise RuntimeError("저장소 루트에서 Notebook을 실행해야 합니다")


PROJECT_ROOT = find_project_root()
MODULE_DIR = PROJECT_ROOT / "examples/06-network-echo"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
from protocol import receive_message, send_message


def echo_once(sock):
    try:
        send_message(sock, receive_message(sock))
    finally:
        sock.close()


server_side, client_side = socket.socketpair()
worker = threading.Thread(target=echo_once, args=(server_side,), daemon=True)
worker.start()
with client_side:
    send_message(client_side, "로컬 Echo")
    echoed = receive_message(client_side)
worker.join(timeout=1)
print("응답:", echoed)


응답: 로컬 Echo


## Checks

한글 메시지와 작업 종료를 확인합니다.


In [2]:
assert echoed == "로컬 Echo"
assert not worker.is_alive()
print("Echo 왕복 검사 통과")


Echo 왕복 검사 통과


## Next Steps

터미널에서는 `echo_server.py`를 먼저 실행한 뒤 별도 창에서 `echo_client.py`를 실행합니다.
